# quantum-cortex — recurrent ladder at 8 seeds (RESUMABLE)

80 units (5 rungs × 8 seeds × 2 tiers), each written to disk the moment it finishes. **A quota cut loses at most one unit.**

## How to resume after a cut
1. This version's Output keeps `metrics/mqar/ladder8-ckpt/` (the finished units).
2. Open a NEW version of this notebook → **Add Input → Your Work → this notebook's previous version** → it mounts under `/kaggle/input/<name>/`.
3. Set `PREV` below to that path (the cell finds it automatically if there is exactly one input).
4. Save & Run All. The runner reuses every finished unit and only runs the rest. Repeat as many times as the quota needs.

GPU T4, Internet On. `--time-budget-min` stops cleanly ~10 min before the 12h session wall; the weekly quota is the real limit — resume across weeks if needed.

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
!rm -rf quantum-cortex
!git clone https://{token}@github.com/vaneeckhoutnicolas/quantum-cortex.git
%cd quantum-cortex
!pip -q install numpy jsonschema

In [ ]:
# Find previous checkpoint dirs mounted as Inputs (resume). Empty list = fresh start.
import glob, os
PREV = sorted(glob.glob('/kaggle/input/*/metrics/mqar/ladder8-ckpt') + glob.glob('/kaggle/input/*/ladder8-ckpt'))
print('resume_from:', PREV if PREV else '(none — fresh start)')
if PREV:
    n = sum(len(glob.glob(p + '/unit-*.json')) for p in PREV)
    print(f'{n} finished units found in previous versions — they will be reused, not recomputed')

In [ ]:
# The run. Units are written to metrics/mqar/ladder8-ckpt/ as they finish.
# --time-budget-min 700 ≈ stop 20 min before the 12h session wall (a quota cut is handled the same way).
resume = ' '.join(f'--resume-from {p}' for p in PREV)
!python -m cortex_eval.resumable_ladder --steps 1500 --seeds 1337 2024 7 42 99 3 11 2026 --time-budget-min 700 {resume}

In [ ]:
# Status: complete (LATEST-ladder8.json) or partial (PROGRESS-ladder8.json)
import json, os
d = 'metrics/mqar/ladder8-ckpt'
if os.path.exists(d + '/LATEST-ladder8.json'):
    r = json.load(open(d + '/LATEST-ladder8.json'))
    print('COMPLETE —', r['units_done'], '/', r['units_total'])
    for rung, st in r['stats'].items(): print(f"  {rung:22s} mean {st['mean']}  CI95 {st['ci95']}")
    print('L4 vs L3:', r['L4_vs_L3'])
    print('\nCommit metrics/mqar/ladder8-ckpt/ (all unit-*.json + LATEST) — the units ARE the record.')
else:
    r = json.load(open(d + '/PROGRESS-ladder8.json'))
    print(f"PARTIAL — {r['units_done']} / {r['units_total']} units. Open a new version with this Output as Input and Run All to resume.")